# 🚗 Train YOLOv8 Car Damage Detection (CarDD)

Notebook ini untuk training model deteksi cacat bodi mobil menggunakan dataset **CarDD** (Car Damage Detection) dan YOLOv8.

**Output:** file `.onnx` yang siap dipakai di project Automotive AI Inspection (on-device, jalan di HP).

---

## Setup
- ⏱️ Estimasi training: **~30-60 menit** (GPU T4 gratis di Colab)
- 📦 Dataset: CarDD (~4000 gambar, 6 kelas cacat)
- 🎯 Model: YOLOv8n (nano) — kecil, cepat di HP

### Cara jalanin:
1. Buka di **Google Colab** (klik badge di bawah)
2. Runtime → Change runtime type → **T4 GPU**
3. Run semua cell satu per satu (Ctrl+F9 untuk run all)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyy0x4/automative-ai-inspection/blob/main/training/train-cardd-yolov8.ipynb)

## 1️⃣ Install Dependencies

In [ ]:
# Install ultralytics (YOLOv8) dan opendatasets (untuk download dari Kaggle)
!pip install ultralytics opendatasets --quiet

import ultralytics
ultralytics.checks()
print("✅ Ultralytics ready")

## 2️⃣ Download Dataset CarDD

Dataset CarDD sudah include label format YOLO. Download via Kaggle (perlu API key).

**Dapat Kaggle API key:**
1. Login ke kaggle.com → Settings → API → Create New Token
2. File `kaggle.json` akan terdownload
3. Saat cell di bawah jalan, upload `kaggle.json` ke Colab

In [ ]:
import opendatasets as od
import os

# Download CarDD YOLO dataset dari Kaggle
# Akan minta upload kaggle.json atau input username+key
dataset_url = 'https://www.kaggle.com/datasets/gabrielfcarvalho/cardd-with-yolo-annotations-images-labels'
od.download(dataset_url)

# Cari folder dataset
base = '/content/cardd-with-yolo-annotations-images-labels'
if not os.path.exists(base):
    base = '/content/cardd-with-yolo-annotations-images-labels/cardd-with-yolo-annotations-images-labels'
print(f"📁 Dataset folder: {base}")
print(os.listdir(base))

## 3️⃣ Cek Dataset & data.yaml

In [ ]:
import yaml

# Baca data.yaml
yaml_path = os.path.join(base, 'data.yaml')
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("📋 data.yaml:")
print(f"   Classes (nc): {data_config['nc']}")
print(f"   Names: {data_config['names']}")
print(f"   Train path: {data_config['train']}")
print(f"   Val path: {data_config['val']}")

# Fix path di data.yaml supaya absolut (biar YOLO bisa temuin)
data_config['train'] = os.path.join(base, 'train', 'images')
data_config['val'] = os.path.join(base, 'val', 'images')
if 'test' in data_config:
    data_config['test'] = os.path.join(base, 'test', 'images')

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)
print("\n✅ Path di data.yaml sudah di-fix ke absolut")

## 4️⃣ Training YOLOv8n

Training model nano (paling kecil & cepat). Hasil terbaik disimpan otomatis di `runs/detect/train/weights/best.pt`.

In [ ]:
from ultralytics import YOLO

# Load YOLOv8n (nano) pretrained weights
model = YOLO('yolov8n.pt')

# Train dengan dataset CarDD
# epochs=50 cukup untuk PoC, naikin ke 100-200 untuk hasil lebih baik
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name='cardd_yolov8n',
    patience=10,       # early stop kalau gak improve 10 epoch
    device=0,          # GPU
    exist_ok=True
)

print("\n🎉 Training selesai!")
print(f"📦 Best weights: runs/detect/cardd_yolov8n/weights/best.pt")

## 5️⃣ Evaluasi Model

In [ ]:
# Validasi di test set
metrics = model.val(data=yaml_path, split='test')
print(f"\n📊 mAP50: {metrics.box.map50:.3f}")
print(f"📊 mAP50-95: {metrics.box.map:.3f}")

## 6️⃣ Coba Inference (Tes Prediksi)

In [ ]:
import glob
from IPython.display import Image, display

# Ambil beberapa gambar test
test_images = glob.glob(os.path.join(base, 'test', 'images', '*'))[:3]

for img_path in test_images:
    results = model.predict(source=img_path, conf=0.3, save=True)
    pred_path = f'/content/runs/detect/predict/{os.path.basename(img_path)}'
    if os.path.exists(pred_path):
        display(Image(filename=pred_path, width=600))
        
        # Print hasil
        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                name = model.names[cls]
                print(f"   🔍 {name} {conf:.0%}")

## 7️⃣ Export ke ONNX ⭐

Ini bagian penting — export model ke format `.onnx` supaya bisa jalan di browser HP (via ONNX Runtime Web).

In [ ]:
# Export best weights ke ONNX
best_weights = 'runs/detect/cardd_yolov8n/weights/best.pt'
model_best = YOLO(best_weights)

export_path = model_best.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    opset=12
)

print(f"\n✅ Model exported!")
print(f"📦 ONNX file: {export_path}")

## 8️⃣ Download ONNX & Plug ke Project

In [ ]:
from google.colab import files

# Download file ONNX ke laptop
files.download(export_path)
print("⬇️ Download dimulai...")

## 9️⃣ Cara Pakai di Project

Setelah dapat file `.onnx`:

1. **Copy file `.onnx`** ke folder `public/models/` di project
2. **Edit 1 file:** `src/services/modelConfig.ts`
   ```ts
   fileName: 'best.onnx',   // nama file kamu
   classes: ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat'],
   ```
3. **Deploy ulang:** `vercel --prod`
4. Buka di HP → model cacat bodi mobil langsung jalan on-device! 🎉

---

### 📌 Catatan Kelas Cacat CarDD:
| Index | Class |
|-------|-------|
| 0 | dent |
| 1 | scratch |
| 2 | crack |
| 3 | glass shatter |
| 4 | lamp broken |
| 5 | tire flat |

Urutan ini **HARUS SAMA** dengan yang di `modelConfig.ts`.